In [1]:
from cuda_cqed.sim import Sim
# import gpu_odes.HatGPUODE_D
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
%matplotlib qt

In [2]:
K = 0.00005
g3 = 4e-3
s1 = 1
g2 = g3*s1

disp = 1

In [3]:
pi = np.pi

sim = Sim(use_complex=True)

sim.add_param('wb', 0e9*2*pi, is_excitation=True)
# sim.add_param('sqrtkb', np.sqrt(1e7 * 2 * np.pi)) # in MHz
sim.add_param('g3', g3)
sim.add_paramsweep('amplG', 0, 1, 101)  # 18 - gain
sim.add_param('IC', disp)
sim.add_param('K', K)

sim.add_EOM('s1', '0', IC_str='amplG')

# sim.add_EOM('b', '-1j*wb*b - (sqrtkb**2/2)*b + 1j*g3*conjugate(b)*s1 - 1j*b*K*abs(b)**2',IC_str='IC')
sim.add_EOM('b', '-2j*K*nb*b - 2j*g3*conjugate(b)*s1',IC_str='IC')
sim.add_EOM('bb', '-2j*K*bb - 4j*K*nb*bb -2j*g3*s1 - 4j*g3*s1*nb', IC_str='IC**2')
sim.add_EOM('nb', '2j*g3*bb*s1 - 2j*g3*conjugate(bb)*s1',IC_str='IC**2')

sim.set_solve_type('all')

sim.specify_time(t_f=200, pts=1001)

sim.validate()

Recent change to specify_time(), check implementation
Simulation validation success!


In [4]:
x, t = sim.solve()

Simulation validation success!
Running GPU solve...


100%|████████████████████████████████████████████████████████████████████████████| 1001/1001 [00:00<00:00, 1683.80it/s]

 
...finished GPU solve!


In [5]:
xd = x.copy()
td = t.copy()

In [6]:
b = x[2,:]+1j*x[3,:]
bb = x[4,:]+1j*x[5,:]
nb = x[6,:]+1j*x[7,:]

In [7]:
def plot_Qfunc(a, aa, na, ax=None, plot_arg=None):

    if ax == None:
        ax = plt.gca()
    
    ad = np.conjugate(a)
    adad = np.conjugate(aa)
    sxx = 1/2 + 1/2*( (aa - a*a) + 2*(na - ad*a) + (adad - ad*ad))
    syy = 1/2 - 1/2*( (aa - a*a) - 2*(na - ad*a) + (adad - ad*ad))
    sxy = np.imag(1/2*((aa - a*a) - (adad - ad*ad)))
    covar =  np.real(np.array([[sxx, sxy], [sxy, syy]]))

    print(covar)
    print(' ')

    phi = np.linspace(0, 2*np.pi, 101)
    
    print(np.argsort(np.linalg.eig(covar)[0]))

    order = np.argsort(np.linalg.eig(covar)[0])
    major_idx = order[1]
    minor_idx = order[0]
    
    theta = -np.angle(np.linalg.eig(covar)[1][:,1][minor_idx]+1j*np.linalg.eig(covar)[1][:,1][major_idx])
    Smajor = np.sort(np.linalg.eig(covar)[0])[major_idx]
    Sminor = np.sort(np.linalg.eig(covar)[0])[minor_idx]
    
    x1 = (Smajor)**(1/4)*np.cos(phi)*2
    y1 = (Sminor)**(1/4)*np.sin(phi)*2
    
    x2 = x1*np.cos(theta) + y1*np.sin(theta) + np.real(a)
    y2 = -x1*np.sin(theta) + y1*np.cos(theta) + np.imag(a)

    if plot_arg == None:
        ax.plot(x2,y2)
    else:
        ax.plot(x2,y2,plot_arg)
    # ax.xlim([-7,7])
    # ax.ylim([-7,7])
    # plt.gca().set_aspect('equal')
    # plt.grid()
    # plt.show()


In [27]:
i = -1
j = -1
print(b[i,j], bb[i,j], nb[i,j])
plot_Qfunc(b[i,j], bb[i,j], nb[i,j])
plt.xlim([-7,7])
plt.ylim([-7,7])
plt.gca().set_aspect('equal')
plt.grid()
plt.show()

(2.465813428403528-2.543723432358023j) (-1.1803283602736017-18.667066935844474j) (18.2377309999211-8.439112014427884e-20j)
[[ 5.39693091 -6.12237214]
 [-6.12237214  6.97700156]]
 
[0 1]


In [9]:
sim.paramsweep_dict['amplG']

array([0.  , 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1 ,
       0.11, 0.12, 0.13, 0.14, 0.15, 0.16, 0.17, 0.18, 0.19, 0.2 , 0.21,
       0.22, 0.23, 0.24, 0.25, 0.26, 0.27, 0.28, 0.29, 0.3 , 0.31, 0.32,
       0.33, 0.34, 0.35, 0.36, 0.37, 0.38, 0.39, 0.4 , 0.41, 0.42, 0.43,
       0.44, 0.45, 0.46, 0.47, 0.48, 0.49, 0.5 , 0.51, 0.52, 0.53, 0.54,
       0.55, 0.56, 0.57, 0.58, 0.59, 0.6 , 0.61, 0.62, 0.63, 0.64, 0.65,
       0.66, 0.67, 0.68, 0.69, 0.7 , 0.71, 0.72, 0.73, 0.74, 0.75, 0.76,
       0.77, 0.78, 0.79, 0.8 , 0.81, 0.82, 0.83, 0.84, 0.85, 0.86, 0.87,
       0.88, 0.89, 0.9 , 0.91, 0.92, 0.93, 0.94, 0.95, 0.96, 0.97, 0.98,
       0.99, 1.  ])

In [23]:
from cuda_cqed.cumulant_plot.cumulant_slider_plot import cumulant_slider_plot

axes_dict = {'amplG': sim.paramsweep_dict['amplG'], 'time': t[0,:]}

In [24]:
cumulant_slider_plot(b, bb, nb, axes_dict)

[[ 4.99999999e-01 -1.99800195e-05]
 [-1.99800195e-05  5.00000001e-01]]
 
[0 1]


[0, 1]
[[ 4.99999996e-01 -3.99600387e-05]
 [-3.99600387e-05  5.00000004e-01]]
 
[0 1]
[0, 0]
[[ 4.99999999e-01 -1.99800195e-05]
 [-1.99800195e-05  5.00000001e-01]]
 
[0 1]
[0, 1]
[[ 4.99999996e-01 -3.99600387e-05]
 [-3.99600387e-05  5.00000004e-01]]
 
[0 1]
[0, 2]
[[ 4.99999991e-01 -5.99400577e-05]
 [-5.99400577e-05  5.00000009e-01]]
 
[0 1]
[0, 1]
[[ 4.99999996e-01 -3.99600387e-05]
 [-3.99600387e-05  5.00000004e-01]]
 
[0 1]
[0, 2]
[[ 4.99999991e-01 -5.99400577e-05]
 [-5.99400577e-05  5.00000009e-01]]
 
[0 1]
[0, 3]
[[ 4.99999984e-01 -7.99200763e-05]
 [-7.99200763e-05  5.00000016e-01]]
 
[0 1]
[0, 4]
[[ 4.99999975e-01 -9.99000942e-05]
 [-9.99000942e-05  5.00000025e-01]]
 
[0 1]
[1, 4]
[[ 4.99999969e-01 -1.79820176e-04]
 [-1.79820176e-04  5.00000059e-01]]
 
[0 1]
[2, 4]
[[ 4.99999977e-01 -2.59740261e-04]
 [-2.59740261e-04  5.00000106e-01]]
 
[0 1]
[3, 4]
[[ 4.99999997e-01 -3.39660351e-04]
 [-3.39660351e-04  5.00000166e-01]]
 
[0 1]
[4, 4]
[[ 5.00000029e-01 -4.19580448e-04]
 [-4.1958044

In [12]:
plt.plot([1,2],[3,4])
plt.show()

In [13]:
b

array([[1.        -1.99800195e-05j, 1.        -3.99600389e-05j,
        1.        -5.99400584e-05j, ..., 0.99980081-1.99587141e-02j,
        0.99980041-1.99786902e-02j, 0.99980001-1.99986662e-02j],
       [1.        -3.59640362e-05j, 1.        -7.19280725e-05j,
        1.        -1.07892109e-04j, ..., 0.99992797-3.59386609e-02j,
        0.99992782-3.59746568e-02j, 0.99992768-3.60106527e-02j],
       [1.        -5.19480530e-05j, 1.        -1.03896106e-04j,
        1.        -1.55844160e-04j, ..., 1.00030958-5.19345605e-02j,
        1.0003102 -5.19866241e-02j, 1.00031082-5.20386880e-02j],
       ...,
       [1.00000123-1.58641436e-03j, 1.00000491-3.17283291e-03j,
        1.00001104-4.75925984e-03j, ..., 2.39535341-2.44255560e+00j,
        2.39835254-2.44712436e+00j, 2.4013546 -2.45170147e+00j],
       [1.00000125-1.60239840e-03j, 1.00000501-3.20480111e-03j,
        1.00001127-4.80721245e-03j, ..., 2.42723954-2.48791621e+00j,
        2.43030615-2.49261199e+00j, 2.4333757 -2.49731648e+00j]

In [14]:
from typing import Union, List, Callable, Tuple, Dict
def _indexData(data: Union[List, np.array], dim: Union[List, np.array]):
    """ get the data at dimension 'dim' of the input data. Basically just make
        the list data can be indexed like np.array data.

    :param data: input data
    :param dim: list of indexs for each dimension
    :return:
    """
    d = data
    for i in dim:
        d = d[int(i)]
    return d

In [15]:
np.shape(b)

(101, 1001)

In [16]:
idx = tuple([3,4])

In [17]:
b[idx]

np.complex128(1.0000000237524724-0.00033966035934947843j)

[3]
[[[ 5.00000000e-01  4.99999999e-01  4.99999999e-01 ...  4.99856591e-01
    4.99856287e-01  4.99855984e-01]
  [-6.79320697e-05 -1.35864140e-04 -2.03796210e-04 ... -6.78819215e-02
   -6.79499062e-02 -6.80178910e-02]]

 [[-6.79320697e-05 -1.35864140e-04 -2.03796210e-04 ... -6.78819215e-02
   -6.79499062e-02 -6.80178910e-02]
  [ 5.00000007e-01  5.00000027e-01  5.00000060e-01 ...  5.06646209e-01
    5.06659539e-01  5.06672882e-01]]]
 
[4]
[[[ 5.00000001e-01  5.00000005e-01  5.00000011e-01 ...  5.01156785e-01
    5.01159073e-01  5.01161363e-01]
  [-8.39160866e-05 -1.67832174e-04 -2.51748263e-04 ... -8.39536185e-02
   -8.40378981e-02 -8.41221784e-02]]

 [[-8.39160866e-05 -1.67832174e-04 -2.51748263e-04 ... -8.39536185e-02
   -8.40378981e-02 -8.41221784e-02]
  [ 5.00000010e-01  5.00000038e-01  5.00000086e-01 ...  5.09560068e-01
    5.09579261e-01  5.09598474e-01]]]
 
[5]
[[[ 5.00000003e-01  5.00000012e-01  5.00000027e-01 ...  5.02965994e-01
    5.02971890e-01  5.02977792e-01]
  [-9.9900103

Traceback (most recent call last):
  File "C:\Users\boris\anaconda3\envs\fullconda\Lib\site-packages\matplotlib\cbook.py", line 361, in process
    func(*args, **kwargs)
  File "C:\Users\boris\anaconda3\envs\fullconda\Lib\site-packages\matplotlib\widgets.py", line 592, in <lambda>
    return self._observers.connect('changed', lambda val: func(val))
                                                          ^^^^^^^^^
  File "c:\users\boris\documents\github\cuda-cqed\cuda_cqed\cumulant_plot\cumulant_slider_plot.py", line 72, in update
    plot_Qfunc(new_a, new_aa, new_na, ax=None, plot_arg=None)
  File "c:\users\boris\documents\github\cuda-cqed\cuda_cqed\cumulant_plot\cumulant_slider_plot.py", line 100, in plot_Qfunc
    print(np.argsort(np.linalg.eig(covar)[0]))
                     ^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\boris\anaconda3\envs\fullconda\Lib\site-packages\numpy\linalg\_linalg.py", line 1481, in eig
    _assert_stacked_square(a)
  File "C:\Users\boris\anaconda3\envs\fullconda

In [18]:
b[5]

array([1.        -9.99001035e-05j, 1.00000001-1.99800208e-04j,
       1.00000003-2.99700317e-04j, ..., 1.00297995-1.00058997e-01j,
       1.00298591-1.00159676e-01j, 1.00299186-1.00260356e-01j])

In [25]:
ax = plt.subplot(1,1,1)

In [26]:
dir(ax)

['ArtistList',
 '_AxesBase__clear',
 '_PROPERTIES_EXCLUDED_FROM_SET',
 '__annotations__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_add_text',
 '_adjustable',
 '_agg_filter',
 '_alias_map',
 '_alpha',
 '_anchor',
 '_animated',
 '_aspect',
 '_autotitlepos',
 '_axes',
 '_axes_locator',
 '_axis_map',
 '_axis_names',
 '_axisbelow',
 '_box_aspect',
 '_callbacks',
 '_check_no_units',
 '_children',
 '_clipon',
 '_clippath',
 '_cm_set',
 '_colorbars',
 '_convert_dx',
 '_current_image',
 '_different_canvas',
 '_errorevery_to_mask',
 '_facecolor',
 '_fill_between_process_units',
 '_fill_between_x_or_y',
 '_forward_navigation_events',
 '_

In [29]:
plt.gca().clear()